In [1]:
# --- Parámetros ---
task_id_param = 24 # Reemplazar con el parámetro del pipeline: notebookutils.widgets.get("task_id")

StatementMeta(, 388c4086-cffa-493a-aea2-6a45825146f2, 3, Finished, Available, Finished)

In [2]:
from pyspark.sql.functions import row_number, col, lit, coalesce, max as spark_max
from datetime import datetime
from pyspark.sql.window import Window
from notebookutils import mssparkutils
from functools import reduce
import json
import traceback
import time
import random

control_table_full_name = "lh_control.dbo.silver_to_gold_control"
table_type_filter = "Security"

# ---- flags performance ----
CALCULATE_WATERMARK = False   # (quita un job)
LOG_ROW_COUNTS = False        # False = no hace count() (ahorra MUCHO)
CACHE_SOURCE = False          # False = no cachea
CACHE_DIMS = False            # False = no cachea dims
USE_WINDOW_DEDUPE = True      # True = usa watermark; False = dropDuplicates (más rápido)

# ---- Control de concurrencia ----
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_delta_operation_with_retry(fn, operation_name: str = "operación Delta") -> None:
    """Ejecuta una operación Delta (write, MERGE, etc.) con reintentos ante ConcurrentAppendException."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fn()
            if attempt > 1:
                print(f"   {operation_name} completada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada en {operation_name} (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

def update_task_status(task_id, status, message, new_watermark=None):
    """Actualiza la tabla de control (con reintentos ante concurrencia)."""
    safe_message = (message or "").replace("'", "''")
    watermark_update_sql = ""
    if new_watermark is not None:
        watermark_str = (
            new_watermark.strftime('%Y-%m-%d %H:%M:%S.%f')
            if isinstance(new_watermark, datetime)
            else str(new_watermark)
        )
        watermark_update_sql = f", last_watermark_value = '{watermark_str}'"

    current_utc_timestamp = datetime.utcnow()
    update_query = f"""
        UPDATE {control_table_full_name}
        SET
            last_run_status = '{status}',
            last_message = '{safe_message}',
            last_run_at = CAST('{current_utc_timestamp}' AS TIMESTAMP),
            updated_at = CAST('{current_utc_timestamp}' AS TIMESTAMP)
            {watermark_update_sql}
        WHERE task_id = {task_id}
    """

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            print(f"[TASK {task_id}] Log actualizado: Estado='{status}'" + (f" (intento {attempt})" if attempt > 1 else ""))
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"[TASK {task_id}] Concurrencia detectada al actualizar control (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"[TASK {task_id}] FATAL: No se pudo actualizar la tabla de control. Razón: {e}")
                raise
    if last_error is not None:
        raise last_error

def process_task(config_row):
    config = config_row.asDict(recursive=True) if hasattr(config_row, "asDict") else config_row

    task_id = config["task_id"]
    source_df = None

    print("\n" + "=" * 80)
    print(f"=== Iniciando Carga FULL (Security) | Task ID: {task_id} ===")
    print("=" * 80)

    try:
        source_object_full_name = f"{config['source_lakehouse']}.{config['source_schema']}.{config['source_table']}"
        target_table_full_name  = f"{config['target_lakehouse']}.{config['target_schema']}.{config['target_table']}"

        business_keys_list = [k.strip() for k in (config.get("business_keys") or "").split(",") if k.strip()]
        watermark_column   = (config.get("watermark_column") or "").strip()
        mappings           = json.loads(config.get("column_mapping_json") or "[]")

        print(f"[TASK {task_id}] Origen  : {source_object_full_name}")
        print(f"[TASK {task_id}] Destino : {target_table_full_name}")

        # 1) Leer origen
        source_df = spark.sql(f"SELECT * FROM {source_object_full_name}")

        # isEmpty barato (evita rdd.isEmpty())
        if not source_df.take(1):
            msg = "No hay registros en la tabla origen (FULL)."
            print(f"[TASK {task_id}] {msg}")
            update_task_status(task_id, "Success", msg)
            return

        if CACHE_SOURCE:
            source_df = source_df.cache()

        # 2) watermark informativo (opcional)
        new_watermark = None
        if CALCULATE_WATERMARK and watermark_column and watermark_column in source_df.columns:
            new_watermark = source_df.select(spark_max(col(watermark_column)).alias("mx")).collect()[0]["mx"]

        # 3) Lookups
        transformed_df = source_df
        lookups = [m for m in mappings if m.get("Type") == "LOOKUP"]

        for i, lookup in enumerate(lookups):
            dim_alias = f"d{i}"
            source_parts = (lookup["Source"] or "").split(";")
            dim_table_str = source_parts[0]

            if len(source_parts) == 2:
                fact_keys = [k.strip() for k in source_parts[1].split(",")]
                dim_keys = fact_keys
            elif len(source_parts) == 3:
                fact_keys = [k.strip() for k in source_parts[1].split(",")]
                dim_keys  = [k.strip() for k in source_parts[2].split(",")]
            else:
                raise ValueError(f"Formato lookup incorrecto: {lookup['Source']}")

            dim_table_full_name = f"{config['target_lakehouse']}.{dim_table_str}"
            surrogate_key = lookup["Target"]

            # leer solo columnas necesarias de dim (más rápido)
            dim_cols = list(set(dim_keys + [surrogate_key]))
            dim_df = spark.table(dim_table_full_name).select(*dim_cols)
            if CACHE_DIMS:
                dim_df = dim_df.cache()

            join_conditions = [
                transformed_df[fact_keys[j]] == col(f"{dim_alias}.{dim_keys[j]}")
                for j in range(len(fact_keys))
            ]
            join_condition = reduce(lambda a, b: a & b, join_conditions)

            df_before = transformed_df
            joined = df_before.join(dim_df.alias(dim_alias), join_condition, "left_outer")

            transformed_df = joined.select(
                *[df_before[c] for c in df_before.columns],
                coalesce(col(f"{dim_alias}.{surrogate_key}"), lit(-1)).alias(surrogate_key)
            )

        # 4) select final
        final_select_cols = []
        for m in mappings:
            if m["Type"] == "LOOKUP":
                final_select_cols.append(col(m["Target"]))
            else:
                final_select_cols.append(col(m["Source"]).alias(m["Target"]))

        final_df = transformed_df.select(*final_select_cols)

        # 5) crear destino si no existe
        if not spark.catalog.tableExists(target_table_full_name):
            final_df.limit(0).write.format("delta").saveAsTable(target_table_full_name)

        # 6) dedupe (elige modo)
        if not business_keys_list:
            raise ValueError("business_keys vacío; no se puede deduplicar.")

        if USE_WINDOW_DEDUPE and watermark_column and watermark_column in final_df.columns:
            window_spec = Window.partitionBy(*business_keys_list).orderBy(col(watermark_column).desc())
            dedup_df = (final_df
                        .withColumn("row_num", row_number().over(window_spec))
                        .filter(col("row_num") == 1)
                        .drop("row_num"))
        else:
            # MUCHÍSIMO más rápido (sin shuffle grande por orderBy watermark)
            dedup_df = final_df.dropDuplicates(business_keys_list)

        # 7) overwrite (FULL)
        def _do_overwrite():
            (dedup_df.write.format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "false")
                .saveAsTable(target_table_full_name))
        run_delta_operation_with_retry(_do_overwrite, f"Overwrite en {target_table_full_name}")

        msg = "Task completed successfully (FULL LOAD)."
        if LOG_ROW_COUNTS:
            msg += f" Rows written: {dedup_df.count()}"

        update_task_status(task_id, "Success", msg, new_watermark)
        print(f"[TASK {task_id}] SUCCESS")

    except Exception as e:
        err = str(e).replace("\n", " ").replace("\r", "")
        print(f"[TASK {task_id}] FATAL ERROR: {err}")
        print(traceback.format_exc())
        update_task_status(task_id, "Failed", err)
        raise

    finally:
        if source_df is not None and CACHE_SOURCE:
            try:
                source_df.unpersist()
            except Exception:
                pass
        print(f"[TASK {task_id}] --- Proceso Finalizado ---")


# ---------------- MAIN ----------------
print(f"--- Buscando tasks {table_type_filter} ---")

tasks_df = spark.sql(f"""
    SELECT *
    FROM {control_table_full_name}
    WHERE is_enabled = true
      AND table_type = '{table_type_filter}'
      AND task_id = {task_id_param}
""")

# isEmpty barato
if not tasks_df.take(1):
    print("No tasks to process.")
    mssparkutils.notebook.exit("No tasks to process.")

tasks = tasks_df.collect()
print(f"Se encontraron {len(tasks)} tasks.")

failed_tasks = []

for config in tasks:
    task_id = config["task_id"]
    try:
        process_task(config)
    except Exception as e:
        failed_tasks.append((task_id, str(e)))
        print(f"[TASK {task_id}] Falló, seguimos con la siguiente.")

if failed_tasks:
    print("Algunas tasks fallaron:")
    for t in failed_tasks:
        print(f" - Task {t[0]}: {t[1]}")
    raise RuntimeError(f"Fallaron {len(failed_tasks)} tarea(s) de tipo '{table_type_filter}'.")
else:
    print("Todas las tasks terminaron correctamente.")


StatementMeta(, 388c4086-cffa-493a-aea2-6a45825146f2, 4, Finished, Available, Finished)

--- Buscando tasks Security ---
Se encontraron 1 tasks.

=== Iniciando Carga FULL (Security) | Task ID: 24 ===
[TASK 24] Origen  : lh_silver_shortcuts.dbo.sec_plants
[TASK 24] Destino : lh_gold_master_data.security.sec_plants
[TASK 24] Log actualizado: Estado='Success'
[TASK 24] SUCCESS
[TASK 24] --- Proceso Finalizado ---
Todas las tasks terminaron correctamente.
